# Feature Engineering

Builds one modelling table with **one row per applicant** by aggregating each source table to `SK_ID_CURR` and LEFT JOINing it to `application_train`. Every feature is based on findings from the data understanding and exploration notebooks.

Output: `data/processed/model_features.parquet` (307,511 rows, default rate 8.07%).

In [1]:
import duckdb
import pandas as pd

APP    = 'data/raw/application_train.csv'
BUREAU = 'data/raw/bureau.csv'
INST   = 'data/raw/installments_payments.csv'
PREV   = 'data/raw/previous_application.csv'

## 1. Application Features

In [2]:
duckdb.sql(f'''
    CREATE OR REPLACE TABLE app_features AS
    SELECT
        SK_ID_CURR,
        TARGET,
        NAME_CONTRACT_TYPE,
        NAME_INCOME_TYPE,
        NAME_EDUCATION_TYPE,
        CASE WHEN FLAG_OWN_CAR    = 'Y' THEN 1 ELSE 0 END AS owns_car,
        CASE WHEN FLAG_OWN_REALTY = 'Y' THEN 1 ELSE 0 END AS owns_realty,
        AMT_INCOME_TOTAL,
        AMT_CREDIT,
        AMT_ANNUITY,
        AMT_CREDIT  / NULLIF(AMT_INCOME_TOTAL, 0) AS credit_to_income,
        AMT_ANNUITY / NULLIF(AMT_INCOME_TOTAL, 0) AS annuity_to_income,
        -DAYS_BIRTH / 365.25 AS age_years,
        CASE WHEN DAYS_EMPLOYED = 365243 THEN NULL ELSE -DAYS_EMPLOYED / 365.25 END AS years_employed,
        CASE WHEN DAYS_EMPLOYED = 365243 THEN 1 ELSE 0 END AS not_employed,
        EXT_SOURCE_1,
        EXT_SOURCE_2,
        EXT_SOURCE_3
    FROM '{APP}'
''')

In [3]:
duckdb.sql("SELECT COUNT(*) AS rows, COUNT(DISTINCT SK_ID_CURR) AS applicants, AVG(not_employed) AS share_not_employed FROM app_features").show()

┌────────┬────────────┬─────────────────────┐
│  rows  │ applicants │ share_not_employed  │
│ int64  │   int64    │       double        │
├────────┼────────────┼─────────────────────┤
│ 307511 │     307511 │ 0.18007160719453938 │
└────────┴────────────┴─────────────────────┘



## 2. Bureau Features

In [4]:
duckdb.sql(f'''
    CREATE OR REPLACE TABLE bureau_features AS
    SELECT
        SK_ID_CURR,
        COUNT(*) AS bureau_num_loans,
        SUM(CASE WHEN CREDIT_ACTIVE = 'Active' THEN 1 ELSE 0 END)                AS bureau_num_active,
        SUM(CASE WHEN CREDIT_ACTIVE IN ('Sold', 'Bad debt') THEN 1 ELSE 0 END)   AS bureau_num_bad,
        MAX(CASE WHEN CREDIT_DAY_OVERDUE > 0 OR AMT_CREDIT_MAX_OVERDUE > 0
                 THEN 1 ELSE 0 END)                                              AS bureau_ever_late,
        MAX(CREDIT_DAY_OVERDUE)                                                  AS bureau_max_days_overdue_now,
        SUM(CASE WHEN CREDIT_ACTIVE = 'Active' AND AMT_CREDIT_SUM_DEBT >= 0
                 THEN AMT_CREDIT_SUM_DEBT END)                                   AS bureau_active_debt
    FROM '{BUREAU}'
    GROUP BY SK_ID_CURR
''')

In [5]:
duckdb.sql("SELECT COUNT(*) AS rows, COUNT(DISTINCT SK_ID_CURR) AS customers, AVG(bureau_ever_late) AS share_ever_late FROM bureau_features").show()

┌────────┬───────────┬─────────────────────┐
│  rows  │ customers │   share_ever_late   │
│ int64  │   int64   │       double        │
├────────┼───────────┼─────────────────────┤
│ 305811 │    305811 │ 0.26952921902743854 │
└────────┴───────────┴─────────────────────┘



## 3. Payment Features (installments)

In [6]:
duckdb.sql(f'''
    CREATE OR REPLACE TABLE payment_features AS
    WITH instalments AS (
        SELECT
            SK_ID_CURR,
            SK_ID_PREV,
            NUM_INSTALMENT_NUMBER,
            MIN(DAYS_INSTALMENT)          AS due_day,
            MAX(DAYS_ENTRY_PAYMENT)       AS last_payment_day,
            MAX(COALESCE(AMT_PAYMENT, 0)) AS largest_payment
        FROM '{INST}'
        GROUP BY SK_ID_CURR, SK_ID_PREV, NUM_INSTALMENT_NUMBER
    )
    SELECT
        SK_ID_CURR,
        COUNT(*) AS pay_num_instalments,
        SUM(CASE WHEN largest_payment = 0 THEN 1 ELSE 0 END) AS pay_num_missed,
        SUM(CASE WHEN largest_payment > 0 AND last_payment_day > due_day THEN 1 ELSE 0 END) AS pay_num_late,
        AVG(CASE WHEN largest_payment > 0 AND last_payment_day > due_day THEN 1.0 ELSE 0.0 END) AS pay_share_late,
        AVG(CASE WHEN largest_payment > 0 THEN GREATEST(last_payment_day - due_day, 0) END) AS pay_avg_days_late,
        MAX(CASE WHEN largest_payment > 0 THEN last_payment_day - due_day END) AS pay_max_days_late
    FROM instalments
    GROUP BY SK_ID_CURR
''')

In [7]:
duckdb.sql("SELECT COUNT(*) AS rows, COUNT(DISTINCT SK_ID_CURR) AS customers, AVG(pay_share_late) AS avg_share_late, MAX(pay_max_days_late) AS worst_days_late FROM payment_features").show()

┌────────┬───────────┬─────────────────────┬─────────────────┐
│  rows  │ customers │   avg_share_late    │ worst_days_late │
│ int64  │   int64   │       double        │     double      │
├────────┼───────────┼─────────────────────┼─────────────────┤
│ 339587 │    339587 │ 0.08466264029975178 │          2884.0 │
└────────┴───────────┴─────────────────────┴─────────────────┘



## 4. Previous Application Features

In [8]:
duckdb.sql(f'''
    CREATE OR REPLACE TABLE prev_features AS
    SELECT
        SK_ID_CURR,
        COUNT(*) AS prev_num_applications,
        SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Approved' THEN 1 ELSE 0 END) AS prev_num_approved,
        SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Refused'  THEN 1 ELSE 0 END) AS prev_num_refused,
        AVG(CASE WHEN NAME_CONTRACT_STATUS = 'Refused'  THEN 1.0 ELSE 0.0 END) AS prev_refusal_rate,
        AVG(CASE WHEN NAME_CONTRACT_STATUS = 'Approved'
                 THEN AMT_CREDIT / NULLIF(AMT_APPLICATION, 0) END) AS prev_avg_granted_vs_asked
    FROM '{PREV}'
    GROUP BY SK_ID_CURR
''')

In [9]:
duckdb.sql("SELECT COUNT(*) AS rows, COUNT(DISTINCT SK_ID_CURR) AS customers, AVG(prev_refusal_rate) AS avg_refusal_rate FROM prev_features").show()

┌────────┬───────────┬─────────────────────┐
│  rows  │ customers │  avg_refusal_rate   │
│ int64  │   int64   │       double        │
├────────┼───────────┼─────────────────────┤
│ 338857 │    338857 │ 0.11142006457364556 │
└────────┴───────────┴─────────────────────┘



## Joining everything

In [11]:
duckdb.sql('''
    CREATE OR REPLACE TABLE model_features AS
    SELECT
        a.*,
        CASE WHEN b.SK_ID_CURR IS NULL THEN 0 ELSE 1 END AS has_bureau_record,
        CASE WHEN p.SK_ID_CURR IS NULL THEN 0 ELSE 1 END AS has_payment_history,
        CASE WHEN v.SK_ID_CURR IS NULL THEN 0 ELSE 1 END AS has_prev_application,
        b.* EXCLUDE (SK_ID_CURR),
        p.* EXCLUDE (SK_ID_CURR),
        v.* EXCLUDE (SK_ID_CURR)
    FROM app_features AS a
    LEFT JOIN bureau_features  AS b ON a.SK_ID_CURR = b.SK_ID_CURR
    LEFT JOIN payment_features AS p ON a.SK_ID_CURR = p.SK_ID_CURR
    LEFT JOIN prev_features    AS v ON a.SK_ID_CURR = v.SK_ID_CURR
''')

In [12]:
import os
os.makedirs('data/processed', exist_ok=True)

duckdb.sql("COPY model_features TO 'data/processed/model_features.parquet' (FORMAT PARQUET)")

In [13]:
duckdb.sql('''
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT SK_ID_CURR) AS applicants,
        ROUND(100.0 * AVG(TARGET), 2) AS default_rate,
        AVG(has_bureau_record) AS share_bureau,
        AVG(has_payment_history) AS share_payment,
        AVG(has_prev_application) AS share_prev
    FROM model_features
''').show()

┌────────┬────────────┬──────────────┬────────────────────┬────────────────────┬────────────────────┐
│  rows  │ applicants │ default_rate │    share_bureau    │   share_payment    │     share_prev     │
│ int64  │   int64    │    double    │       double       │       double       │       double       │
├────────┼────────────┼──────────────┼────────────────────┼────────────────────┼────────────────────┤
│ 307511 │     307511 │         8.07 │ 0.8568506492450677 │ 0.9483985938714388 │ 0.9464929709831518 │
└────────┴────────────┴──────────────┴────────────────────┴────────────────────┴────────────────────┘



## Feature summary

| Source | Features | Notes |
|---|---|---|
| Application | contract type, income type, education, owns car / realty, income, credit, annuity, credit-to-income, annuity-to-income, age, years employed, not employed, `EXT_SOURCE_1/2/3` | `DAYS_EMPLOYED = 365243` set to NULL with a `not_employed` flag; Y/N converted to 1/0; gender excluded as a sensitive attribute |
| Bureau | number of loans, active loans, bad loans (sold / bad debt), ever late, max days overdue now, active debt | Negative debt values excluded |
| Payments | instalments, missed, late, share late, average days late, max days late | Payments aggregated to instalment level first (split payments, version rows); NULL and 0 payments both treated as missed; early payments count as 0 days late |
| Previous applications | applications, approved, refused, refusal rate, average granted vs asked | Granted vs asked computed on approved applications only |
| Coverage flags | `has_bureau_record`, `has_payment_history`, `has_prev_application` | Absence of history is itself a signal (Q2, Q3, Q6) |

**Checks:** 307,511 rows and unique applicants; default rate unchanged at 8.07%; coverage shares match the group sizes found in exploration (85.69% bureau, 94.84% payment history, 94.65% previous applications).

**Left for modelling:** handling of remaining NULLs, encoding of text categories, capping extreme values (e.g. a maximum lateness of 2,884 days).